# DFT Foundations: grid, density, potentials, SCF

This is the DFT entry notebook.  It keeps the current limits explicit:
orthorhombic cell, plane-wave/grid representation, Γ point by default, and a
toy local pseudopotential path.

The Kohn-Sham equations replace the many-electron wavefunction with auxiliary
one-electron orbitals:

$$
\left[-\frac{1}{2}\nabla^2 + V_\mathrm{loc}(r) + V_H[\rho](r) + V_{xc}[\rho](r)\right]
\psi_i(r)=\epsilon_i\psi_i(r),
$$

with closed-shell density

$$
\rho(r)=2\sum_i |\psi_i(r)|^2.
$$


In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import mlx.core as mx
import numpy as np


def find_repo_root() -> Path:
    current = Path.cwd().resolve()
    for candidate in (current, *current.parents):
        if (candidate / "pyproject.toml").exists():
            return candidate
    raise RuntimeError("could not locate repository root")


ROOT = find_repo_root()


## Grid representation and integration

The real-space grid stores fields such as \(\rho(r)\), \(V(r)\), and
\(\psi_i(r)\) on \(N_xN_yN_z\) points inside an orthorhombic periodic cell.
Integrals become weighted sums:

$$
\int_\Omega f(r)\,dr
\approx \Delta V\sum_g f(r_g),
\qquad
\Delta V = \frac{\Omega}{N_xN_yN_z}.
$$

For normalized orbitals,

$$
\int_\Omega |\psi_i(r)|^2\,dr = 1,
\qquad
\int_\Omega \rho(r)\,dr = N_e.
$$

The reciprocal grid supplies vectors \(G\).  In a plane-wave representation the
kinetic operator is diagonal:

$$
T\psi_G = \frac{1}{2}|G|^2\psi_G.
$$


## Hartree and exchange-correlation pieces

The Hartree potential is the classical electrostatic potential generated by the
electron density:

$$
\nabla^2 V_H(r) = -4\pi\rho(r).
$$

In reciprocal space this becomes

$$
V_H(G)=\frac{4\pi\rho(G)}{|G|^2},\qquad G\ne 0.
$$

The \(G=0\) term is set to zero in this periodic toy implementation, which
chooses a reference for the average electrostatic potential.

For exchange-correlation, this notebook uses an LDA-style functional:

$$
E_{xc}[\rho] = \int_\Omega \rho(r)\,
\epsilon_{xc}(\rho(r))\,dr,
\qquad
V_{xc}(r)=\frac{\delta E_{xc}}{\delta\rho(r)}.
$$


## Build and solve a two-center toy system

The SCF loop repeatedly builds the effective potential from the current density,
solves the Kohn-Sham eigenproblem, rebuilds the density, and mixes it with the
previous density.


### SCF fixed-point iteration

The Kohn-Sham map takes an input density and returns a new output density:

$$
\rho_\mathrm{out}
= \mathcal F[\rho_\mathrm{in}].
$$

Simple linear mixing forms the next input density as

$$
\rho_{n+1}
= (1-\alpha)\rho_n+\alpha\rho_\mathrm{out}.
$$

DIIS/Pulay mixing uses a short history of residuals to extrapolate a better
input density.  The residual plotted later is

$$
R_n = \|\rho_\mathrm{out}-\rho_\mathrm{in}\|.
$$


In [ ]:
from mlx_atomistic.dft import DFTSystem, LDAExchangeCorrelation, SCFConfig, run_scf

system = DFTSystem.two_center(
    cell=(8.0, 8.0, 8.0),
    grid_shape=(8, 8, 8),
    centers=((3.3, 4.0, 4.0), (4.9, 4.0, 4.0)),
    electron_count=2.0,
    amplitudes=(-2.0, -2.0),
    widths=(0.85, 0.85),
    charges=(0.7, 0.7),
)
config = SCFConfig(
    max_iterations=12,
    tolerance=1e-6,
    mixing=0.35,
    solver="dense",
    mixer="diis",
    convergence_mode="either",
    seed=4,
)
result = run_scf(system, config=config, xc_functional=LDAExchangeCorrelation())
print(result.to_dict() | {"history": f"{len(result.history)} iterations"})


## Density and potential slices

A scalar density on a 3D grid is easier to reason about with slices.  The
mid-plane below shows where the electron density sits relative to the effective
potential landscape.


### What to look for in the slice

For this two-center toy system the density should concentrate near the two
attractive local wells.  The potential slice combines several terms:

$$
V_\mathrm{eff}(r)
= V_\mathrm{loc}(r)+V_H[\rho](r)+V_{xc}[\rho](r).
$$

The potential and density do not need to have the same visual shape.  The
density is determined by the occupied eigenvectors of the Hamiltonian built
from that potential.


In [ ]:
density = np.asarray(result.density)
potential = np.asarray(result.effective_potential)
z = density.shape[2] // 2

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
im0 = axes[0].imshow(density[:, :, z].T, origin="lower", cmap="magma")
axes[0].set_title(r"density slice $\rho(x,y,z_\mathrm{mid})$")
fig.colorbar(im0, ax=axes[0], fraction=0.046)
im1 = axes[1].imshow(potential[:, :, z].T, origin="lower", cmap="coolwarm")
axes[1].set_title(r"effective potential slice $V_\mathrm{eff}$")
fig.colorbar(im1, ax=axes[1], fraction=0.046)
fig.tight_layout()


## SCF convergence diagnostics

Energy alone is not enough.  A useful SCF result should expose residuals,
energy deltas, orbital residuals, and orthonormality error.


### Energy consistency checks

The total energy is decomposed so we can see which physical term changed:

$$
E_\mathrm{total}
= E_\mathrm{kinetic}
+ E_\mathrm{local}
+ E_H
+ E_{xc}
+ E_\mathrm{center-center}.
$$

Orbital residuals check whether the reported orbitals solve the eigenproblem:

$$
r_i = \|H\psi_i-\epsilon_i\psi_i\|.
$$

Orthonormality checks whether the occupied orbitals still satisfy

$$
\langle\psi_i|\psi_j\rangle = \delta_{ij}.
$$


In [ ]:
history = result.history
iterations = [row["iteration"] for row in history]
energies = [row["total"] for row in history]
residuals = [row["density_residual"] for row in history]

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].plot(iterations, energies, marker="o")
axes[0].set_title("SCF total energy")
axes[0].set_xlabel("iteration")
axes[0].set_ylabel("energy / Ha")
axes[1].semilogy(iterations, residuals, marker="o")
axes[1].set_title("density residual")
axes[1].set_xlabel("iteration")
axes[1].set_ylabel(r"$||\rho_\mathrm{out}-\rho_\mathrm{in}||$")
fig.tight_layout()

print("energy terms:", result.energy_by_term)
orbital_residuals = (
    None if result.orbital_residuals is None else np.asarray(result.orbital_residuals)
)
print("orbital residuals:", orbital_residuals)
print("orthonormality error:", result.orthonormality_error)
